# Data understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler


In [ ]:

# Load Dataset
df = pd.read_csv('data/raw/breast-cancer.csv')

# Display first 5 rows
df.head()

In [ ]:
# DATASET INFORMATION

## Number of rows and columns  
print("Dataset Shape:", df.shape)

## Display column names
print("Columns:\n", df.columns.tolist())


In [ ]:
## General info about dataset
df.info()

In [ ]:
## Check missing values
print("\nMissing Values:\n", df.isnull().sum())

In [ ]:
## Summary statistics
df.describe()

In [ ]:
# Target Variable Analysis

## Count values in target column (diagnosis)
df['diagnosis'].value_counts()

## Percentage distribution
df['diagnosis'].value_counts(normalize=True) * 100

In [ ]:
# Check for Duplicates
## Rows
print("Duplicate rows:", df.duplicated().sum())

# Outlier detection , removal & visualization

In [ ]:
# drop the id column (not useful for analysis)
df = df.drop(columns=['id'])

# encode diagnosis: M (Malignant) = 1, B (Benign) = 0
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

df.head(3)


#### Detect outliers using the IQR method

In [ ]:
num_col = df.select_dtypes(include=np.number).columns.drop('diagnosis')
print(num_col.tolist())



In [ ]:
#now we check for outliers
q1  = df[num_col].quantile(0.25)
q3  = df[num_col].quantile(0.75)
iqr = q3 - q1

outlier_mask = (df[num_col] < (q1 - 1.5 * iqr)) | (df[num_col] > (q3 + 1.5 * iqr))

outlier_counts = outlier_mask.sum()
print(outlier_counts)

# detect rows where ANY column is an outlier
row_outlier_mask = outlier_mask.any(axis=1)

# total number of outlier rows
outlier_count = row_outlier_mask.sum()
print(outlier_count)

In [ ]:

# before removing outliers
plt.figure(figsize=(18, 6))
df[num_col].boxplot()
plt.title("Before Outlier Removal")
plt.xticks(rotation=45)
plt.show()

# individual boxplots per feature (grid layout)
df[num_col].plot.box(
    subplots=True,
    layout=(5, 6),
    figsize=(20, 18),
    sharey=False,
    title="Boxplots Before Outlier Removal (Per Feature)"
)
plt.tight_layout()
plt.show()

In [ ]:
df_cleaned = df.loc[~row_outlier_mask].copy()

print('Original shape:', df.shape)
print('Cleaned shape :', df_cleaned.shape)
print('Rows removed   :', df.shape[0] - df_cleaned.shape[0])

In [ ]:
# boxplots after removing outlier rows
plt.figure(figsize=(18, 6))
df_cleaned[num_col].boxplot()
plt.title("Boxplots After Outlier Removal")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

df_cleaned[num_col].plot.box(
    subplots=True,
    layout=(5, 6),
    figsize=(20, 18),
    sharey=False,
    title='Boxplots After Removing Outlier Rows (Per Feature)'
)
plt.tight_layout()
plt.show()

### 4.3 Outlier Detection Using Z-Score Method

In [ ]:
# Z-Score Outlier Detection
from scipy import stats

# Calculate Z-scores for numerical columns
z_scores = np.abs(stats.zscore(df[num_col]))

# Define threshold
threshold = 3

# Identify outliers
z_outlier_mask = (z_scores > threshold)



In [ ]:
# Count outliers per column
z_outlier_counts = z_outlier_mask.sum(axis=0)
print("Outliers per column using Z-score method:")
print(z_outlier_counts)

# Total outlier rows
z_row_outlier_mask = z_outlier_mask.any(axis=1)
z_outlier_count = z_row_outlier_mask.sum()
print(f"\nTotal rows with outliers (Z-score): {z_outlier_count}")


In [ ]:
# Visualize Z-scores for features with high outliers vs low outliers
plt.figure(figsize=(14, 6))

# Define high and low outlier features
high_outlier_features = [i for i, count in enumerate(z_outlier_counts) if count >= 10]  # >= 10 outliers
low_outlier_features = [i for i, count in enumerate(z_outlier_counts) if 0 < count < 5]  # 1-4 outliers

# Plot for features with high outliers
plt.subplot(1, 2, 1)
for i in high_outlier_features[:3]:  # Top 3 high outlier features
    plt.hist(z_scores[:, i], bins=20, alpha=0.6, label=f'{num_col[i]} ({z_outlier_counts[i]} outliers)')
plt.axvline(threshold, color='red', linestyle='--', label='Threshold (3)')
plt.title('Z-scores for Features with High Outliers (≥10)')
plt.xlabel('Z-score')
plt.ylabel('Frequency')
plt.legend()

# Plot for features with low outliers
plt.subplot(1, 2, 2)
for i in low_outlier_features[:3]:  # Top 3 low outlier features
    plt.hist(z_scores[:, i], bins=20, alpha=0.6, label=f'{num_col[i]} ({z_outlier_counts[i]} outliers)')
plt.axvline(threshold, color='red', linestyle='--', label='Threshold (3)')
plt.title('Z-scores for Features with Low Outliers (1-4)')
plt.xlabel('Z-score')
plt.ylabel('Frequency')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:

# --- Scaling the data using Z-score (Standardization) ---
# create scaler object
scaler = StandardScaler()

# copy cleaned dataset so original stays unchanged
df_scaled = df_cleaned.copy()

# apply scaling on all numeric columns except the target (diagnosis)
df_scaled[num_col] = scaler.fit_transform(df_cleaned[num_col])

# print few rows to check
print("After scaling:")
print(df_scaled.head())


In [ ]:

# --- Compare before and after scaling ---
sns.set(style="whitegrid")
plt.figure(figsize=(20, 7))

# BEFORE scaling
plt.subplot(1, 2, 1)
sns.boxplot(data=df_cleaned[num_col], orient="h")
plt.title("Before Scaling", fontsize=14)
plt.xlabel("Value")
plt.ylabel("Features")

# AFTER scaling
plt.subplot(1, 2, 2)
sns.boxplot(data=df_scaled[num_col], orient="h")
plt.title("After Z-score Scaling", fontsize=14)
plt.xlabel("Standardized Value")
plt.ylabel("")

plt.tight_layout()
plt.show()